In [30]:
import pandas as pd
import numpy as np
import re  
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, Dense, Dropout, Embedding, Input, LSTM
from tensorflow.keras.callbacks import EarlyStopping

In [31]:
data = pd.read_csv('swiggy.csv')
print("Columns in the dataset:")
print(data.columns.tolist())

Columns in the dataset:
['ID', 'Area', 'City', 'Restaurant Price', 'Avg Rating', 'Total Rating', 'Food Item', 'Food Type', 'Delivery Time', 'Review']


In [32]:
positive_reviews = {
    "absolutely loved it", "amazing taste and quick delivery",
    "best meal ive had in a while", "delicious and fresh",
    "good but nothing extraordinary", "highly recommended",
    "my new favorite dish", "perfectly cooked and wellseasoned",
    "superb packaging and presentation", "tasty and worth the price",
    "would order again if needed"
}
negative_reviews = {
    "cold and stale food", "disappointed", "late delivery ruined it",
    "mediocre experience", "not as described", "not worth the price",
    "terrible taste", "too spicy and oily", "worst experience ever",
    "wouldnt order again"
}

data = data.dropna(subset=["Review"]).copy()
data["Review"] = data["Review"].str.lower().str.replace(
    r'[^a-z0-9\s]', '', regex=True
).str.strip()
data = data[data["Review"].isin(positive_reviews | negative_reviews)].copy()
data["sentiment"] = data["Review"].isin(positive_reviews).astype("int32")

In [33]:
max_features = 5000
max_length = 200

tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(data["Review"])
X = pad_sequences(tokenizer.texts_to_sequences(
    data["Review"]), maxlen=max_length, padding="post", truncating="post")
y = data['sentiment'].values

In [34]:
max_features = 5000
max_length = 200

tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(data["Review"])
X = pad_sequences(tokenizer.texts_to_sequences(
    data["Review"]), maxlen=max_length, padding="post", truncating="post")
y = data['sentiment'].values

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

In [36]:
model = Sequential([
    Input(shape=(max_length,)),
    Embedding(input_dim=max_features, output_dim=64),
    Bidirectional(LSTM(32)),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [37]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, y_val),
    class_weight={
        0: len(y_train) / (2 * np.sum(y_train == 0)),
        1: len(y_train) / (2 * np.sum(y_train == 1))
    },
    callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
    verbose=1
)

score = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {score[1]:.2f}")

Epoch 1/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.7146 - loss: 0.6007 - val_accuracy: 0.7156 - val_loss: 0.5974
Epoch 2/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.7160 - loss: 0.5990 - val_accuracy: 0.7156 - val_loss: 0.5976
Epoch 3/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.7160 - loss: 0.5985 - val_accuracy: 0.7156 - val_loss: 0.5980
Epoch 4/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.7160 - loss: 0.5982 - val_accuracy: 0.7156 - val_loss: 0.5981
Epoch 5/5
180/180 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.7160 - loss: 0.5980 - val_accuracy: 0.7156 - val_loss: 0.5982
Test accuracy: 0.72


In [38]:
def predict_sentiment(review_text):
    text = review_text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)

    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_length, padding="post", truncating="post")

    prediction = model.predict(padded)[0][0]
    return f"{'Positive' if prediction >= 0.5 else 'Negative'} (Probability: {prediction:.2f})"


sample_review = "The food was worst than expected and wate of money."
print(f"Review: {sample_review}")
print(f"Sentiment: {predict_sentiment(sample_review)}")

Review: The food was worst than expected and wate of money.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step
Sentiment: Positive (Probability: 0.74)
